# WavLM + Cross-Entropy Baseline Training
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
# EDIT these paths to match where you stored your data on Drive
DRIVE_DATA_PATH = '/content/drive/MyDrive/SER_data'
SAVE_TO_DRIVE   = '/content/drive/MyDrive/SER_runs'
os.makedirs(SAVE_TO_DRIVE, exist_ok=True)

In [ ]:
!git clone https://github.com/HithaBadikillaya/Speech-Emotion-Recognition.git
%cd Speech-Emotion-Recognition
!git log --oneline -3

In [ ]:
!pip install -q transformers soundfile scipy scikit-learn tensorboard seaborn
print('Done!')

In [ ]:
import os
# Symlink your Drive data folder into the repo's data/ directory
if os.path.exists(DRIVE_DATA_PATH) and not os.path.exists('data'):
    os.symlink(DRIVE_DATA_PATH, 'data')
    print('Linked data from Drive')
elif not os.path.exists(DRIVE_DATA_PATH):
    print(f'WARNING: {DRIVE_DATA_PATH} not found! Edit DRIVE_DATA_PATH above.')
!echo 'Train:';  wc -l data/metadata/train.csv
!echo 'Val:';    wc -l data/metadata/validation.csv

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Sanity check — should complete in ~30 seconds on GPU
!PYTHONPATH=. python src/training/train.py --config configs/wavlm_ce.yaml --dry-run

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

In [ ]:
# Full training — ~1-2 hours on T4 GPU
!PYTHONPATH=. python src/training/train.py --config configs/wavlm_ce.yaml

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt, seaborn as sns
with open('runs/wavlm_ce/results.json') as f:
    r = json.load(f)
print(f"Best UAR  : {r['best_uar']:.4f}")
print(f"Final UAR : {r['final_val_uar']:.4f}")
print(f"WAR (Acc) : {r['final_val_war']:.4f}")
print(f"F1 Macro  : {r['final_val_f1_macro']:.4f}")

EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
cm = np.array(r['confusion_matrix'])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f"WavLM CE — Confusion Matrix (UAR={r['best_uar']:.3f})")
plt.tight_layout()
plt.savefig('runs/wavlm_ce/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
import shutil
dest = os.path.join(SAVE_TO_DRIVE, 'wavlm_ce')
shutil.copytree('runs/wavlm_ce', dest, dirs_exist_ok=True)
print(f'Saved to Drive: {dest}')
!ls {dest}